# CEG-WM disabled-routing content-combination directional diagnosis

This output-free Notebook is the thin Colab entrypoint for one development-only diagnostic. It runs one operational preflight, thirty-two clean cross-fit references, and eight six-image probes. Each probe fixes embedding coefficient `a` separately at 0.25, 0.50, and 0.75 and records C0, C1(w) for w in 0.25, 0.50, 0.75, and C2 without selecting a coefficient, detector weight, or function. The formal detector remains HF-only. A passing result can only request a separate candidate-selection gate; it does not fit a threshold/FPR or support promotion, calibration, evaluation, baseline, joint, or paper claims.


In [ ]:
from google.colab import drive, userdata
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
EXECUTION_REVISION = 'b242261f10a034b541c571afd30b91b77eaddf19'
RUN_ID = 'ceg_wm_content_uniform_combination_directional_diagnosis'
SESSION_ID = datetime.now(timezone.utc).strftime('colab_%Y%m%dt%H%M%S%fz')
CHECKOUT_ROOT = Path(f'/content/ceg_wm_content_uniform_combination_checkout_{SESSION_ID}')
DRIVE_MOUNT = Path('/content/drive').resolve()
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'CEG-WM' / 'content_uniform_combination_directional_diagnosis'
PERSISTENT_ROOT = DRIVE_ROOT / 'persistent'
WHITENING_ASSET_PERSISTENT_ROOT = DRIVE_MOUNT / 'MyDrive' / 'CEG-WM' / 'lf_whitened_score_screening' / 'persistent'
CACHE_ROOT = DRIVE_ROOT / 'cache'
EXPORT_BASE = DRIVE_ROOT / 'exports' / EXECUTION_REVISION / RUN_ID
EXPORT_ROOT = EXPORT_BASE / SESSION_ID
for required_root in (PERSISTENT_ROOT, CACHE_ROOT, EXPORT_BASE):
    required_root.mkdir(parents=True, exist_ok=True)
    assert DRIVE_MOUNT in required_root.resolve().parents
assert WHITENING_ASSET_PERSISTENT_ROOT.is_dir()
probe_path = PERSISTENT_ROOT / f'.write_probe_{SESSION_ID}'
with probe_path.open('x', encoding='utf-8') as probe:
    probe.write('content-combination persistent root available\n')
probe_path.unlink()
secret_environment = os.environ.copy()
secret_environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
secret_environment['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
assert secret_environment['HF_TOKEN'] and secret_environment['CEG_WM_ROOT_KEY']


In [ ]:
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'init'], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'fetch', '--depth', '1', 'origin', EXECUTION_REVISION], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
observed_revision = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
observed_status = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout
assert observed_revision == EXECUTION_REVISION and observed_status == ''


In [ ]:
server_entrypoint = CHECKOUT_ROOT / 'scripts/experiment_execution/content_uniform_combination_directional_diagnosis_server.py'
command = [sys.executable, str(server_entrypoint), '--repository-root', str(CHECKOUT_ROOT), '--expected-revision', EXECUTION_REVISION, '--persistent-root', str(PERSISTENT_ROOT), '--whitening-asset-persistent-root', str(WHITENING_ASSET_PERSISTENT_ROOT), '--cache-root', str(CACHE_ROOT), '--run-id', RUN_ID, '--session-id', SESSION_ID]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=secret_environment)
assert process.stdout is not None
for log_line in process.stdout:
    print(log_line, end='')
server_exit_code = process.wait()
del secret_environment
receipt_source = PERSISTENT_ROOT / RUN_ID / 'server_receipts' / SESSION_ID / 'execution_receipt.json'
assert receipt_source.is_file(), 'server did not persist the execution receipt'
receipt = json.loads(receipt_source.read_text(encoding='utf-8'))
assert receipt['committed_revision'] == EXECUTION_REVISION
assert receipt['run_id'] == RUN_ID and receipt['session_id'] == SESSION_ID
assert (receipt['operational_unit_count'], receipt['reference_fit_cluster_count'], receipt['directional_probe_cluster_count'], receipt['total_unit_count']) == (1, 32, 8, 41)
assert receipt['maximum_attempts_per_unit'] == 1 and receipt['exit_code'] == server_exit_code
if server_exit_code == 0:
    assert receipt['termination_reason'] == 'frozen_roster_complete'
    assert receipt['content_uniform_combination_directional_aggregate'] is not None


In [ ]:
def file_sha256(path):
    digest = sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_to_drive_export(source, destination, expected_sha256):
    source, destination = Path(source), Path(destination)
    if destination.exists():
        raise RuntimeError('Drive export destination already exists')
    shutil.copyfile(source, destination)
    if file_sha256(destination) != expected_sha256:
        raise RuntimeError('Drive export SHA-256 mismatch')
    return destination

artifact_source = Path(receipt['artifact_path']).resolve()
assert artifact_source.is_file() and not artifact_source.is_symlink() and PERSISTENT_ROOT.resolve() in artifact_source.parents
assert receipt['artifact_kind'] in {'content_uniform_combination_directional_diagnosis_result', 'content_uniform_combination_directional_diagnosis_failure'}
assert receipt['formal_tau_created'] is False and receipt['fpr_estimated'] is False
assert receipt['candidate_promoted'] is False and receipt['scientific_claims_supported'] is False
assert file_sha256(artifact_source) == receipt['artifact_sha256']
receipt_sha256 = file_sha256(receipt_source)
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)
artifact_export = copy_to_drive_export(artifact_source, EXPORT_ROOT / artifact_source.name, receipt['artifact_sha256'])
receipt_export = copy_to_drive_export(receipt_source, EXPORT_ROOT / 'execution_receipt.json', receipt_sha256)
checksums_path = EXPORT_ROOT / 'SHA256SUMS'
with checksums_path.open('x', encoding='utf-8') as checksums:
    checksums.write(f"{receipt['artifact_sha256']}  {artifact_export.name}\n")
    checksums.write(f'{receipt_sha256}  {receipt_export.name}\n')
summary = {'artifact_kind': receipt['artifact_kind'], 'artifact_path': str(artifact_export), 'artifact_sha256': receipt['artifact_sha256'], 'receipt_path': str(receipt_export), 'receipt_sha256': receipt_sha256, 'checksums_path': str(checksums_path), 'committed_revision': EXECUTION_REVISION, 'run_id': RUN_ID, 'session_id': SESSION_ID, 'committed_unit_count': receipt.get('committed_unit_count', 0), 'termination_reason': receipt.get('termination_reason'), 'content_uniform_combination_directional_aggregate': receipt.get('content_uniform_combination_directional_aggregate')}
print(json.dumps(summary, indent=2, sort_keys=True))
if server_exit_code != 0:
    raise RuntimeError('Content-combination diagnosis ended with a diagnostic; Drive export is preserved')
